In [2]:
import pandas as pd

user_interactions = pd.read_csv("../data/interaction/user_interactions.csv")

print("Dataset shape:", user_interactions.shape)
print(user_interactions.columns.tolist())
display(user_interactions)

Dataset shape: (9, 3)
['user_id', 'talk_id', 'rating']


,user_id,talk_id,rating
0,U001,995,5
1,U001,1307,4
2,U001,1879,3
3,U002,995,4
4,U002,1879,5
5,U002,41223,2
6,U003,1307,5
7,U003,41223,3
8,U003,32269,4


In [3]:
user_item_matrix = user_interactions.pivot(
    index="user_id",
    columns="talk_id",
    values="rating"
)

display(user_item_matrix)

talk_id,995,1307,1879,32269,41223
user_id,,,,,
U001,5.0,4.0,3.0,NaN,NaN
U002,4.0,NaN,5.0,NaN,2.0
U003,NaN,5.0,NaN,4.0,3.0


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

user_item_filled = user_item_matrix.fillna(0)

similarity_matrix = cosine_similarity(user_item_filled)

user_similarity = pd.DataFrame(
    similarity_matrix,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

display(user_similarity)

user_id,U001,U002,U003
user_id,,,
U001,1.000000,0.737865,0.400000
U002,0.737865,1.000000,0.126491
U003,0.400000,0.126491,1.000000


In [5]:
selected_user = "U001"

similar_users = user_similarity[selected_user].drop(selected_user).sort_values(ascending=False)

print("Similar users for", selected_user)
display(similar_users)

Similar users for U001


user_id
U002    0.737865
U003    0.400000
Name: U001, dtype: float64

In [6]:
def recommend_for_user(user_id, top_n=5):
    similar_users = user_similarity[user_id].drop(user_id).sort_values(ascending=False)

    rated_talks = user_item_matrix.loc[user_id].dropna().index

    recommendations = {}

    for similar_user, similarity_score in similar_users.items():
        for talk_id, rating in user_item_matrix.loc[similar_user].dropna().items():
            if talk_id not in rated_talks:
                if talk_id not in recommendations:
                    recommendations[talk_id] = []

                recommendations[talk_id].append((rating, similarity_score))

    recommendation_scores = {}

    for talk_id, ratings in recommendations.items():
        total_score = sum(rating * similarity for rating, similarity in ratings)
        total_similarity = sum(similarity for rating, similarity in ratings)

        recommendation_scores[talk_id] = total_score / total_similarity

    recommended_talks = pd.Series(recommendation_scores).sort_values(ascending=False)

    return recommended_talks.head(top_n)

In [8]:
for user_id in user_item_matrix.index:
    print(f"\nRecommendations for {user_id}:")
    display(recommend_for_user(user_id))


Recommendations for U001:


32269    4.000000
41223    2.351536
dtype: float64


Recommendations for U002:


1307     4.146341
32269    4.000000
dtype: float64


Recommendations for U003:


995     4.759747
1879    3.480506
dtype: float64

In [9]:
ted_talks = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

for user_id in user_item_matrix.index:
    recommendations = recommend_for_user(user_id).reset_index()
    recommendations.columns = ["talk_id", "predicted_rating"]

    result = recommendations.merge(
        ted_talks[["talk_id", "title"]],
        on="talk_id",
        how="left"
    )

    print(f"\nRecommendations for {user_id}:")
    display(result[["talk_id", "title", "predicted_rating"]])


Recommendations for U001:


,talk_id,title,predicted_rating
0,32269,The uncomplicated truth about women's sexuality,4.000000
1,41223,The mysterious science of pain,2.351536



Recommendations for U002:


,talk_id,title,predicted_rating
0,1307,Medicine for the 99 percent,4.146341
1,32269,The uncomplicated truth about women's sexuality,4.000000



Recommendations for U003:


,talk_id,title,predicted_rating
0,995,Glorious visions in animation and performance,4.759747
1,1879,What is so special about the human brain?,3.480506


Observations:
1. Similar users were identified using cosine similarity based on their rating patterns.
2. Recommendations were generated using ratings from similar users, weighted by their similarity scores.
3. Different users received different recommendations based on their previous ratings.
4. TED Talks already rated by a user were excluded from recommendations.
5. Predicted ratings were used to rank recommended TED Talks.
6. The model has limitations because the interaction dataset contains only three users and five TED Talks. New users without interaction history cannot receive personalized recommendations.